# AX Job Agent Pipeline

잡코리아 AX/AI/데이터 관련 채용공고 수집-분석-보고 파이프라인을 셀 단위로 개발/검증하는 노트북입니다.

진행 상황은 [`docs/PROGRESS.md`](../docs/PROGRESS.md), 전체 사양은 [`docs/SPEC.md`](../docs/SPEC.md) 참고.

# STEP 01. 개발환경 확인

## 작업 계획

본격적인 크롤링/분석 코드를 작성하기 전에, 개발 환경(Python 버전, 필요 패키지)이 정상적으로 준비되었는지 확인한다.

확인 항목:
- Python 버전 및 플랫폼 정보
- pandas, requests, beautifulsoup4 import 및 버전 확인

아직 크롤링이나 데이터 수집 코드는 작성하지 않는다.

In [ ]:
import sys
import platform

print("Python:", sys.version)
print("Platform:", platform.platform())

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

print("pandas:", pd.__version__)
print("requests:", requests.__version__)
print("BeautifulSoup import: OK")

## 실행 결과 해석

- 실행 성공 여부: 성공 (에러 없이 모든 셀 정상 실행)
- Python / 패키지 버전: Python 3.11.9 / pandas 3.0.6, requests 2.34.2, beautifulsoup4 import OK
- 예상과 다른 부분: 처음에 Jupyter 커널이 다른 프로젝트(`python-study`)의 `.venv`로 잘못 연결되어 `ModuleNotFoundError`가 발생했음. `Select Interpreter`에서 `Workspace` 표시가 붙은 `.\.venv\Scripts\python.exe`로 재선택 + 커널 재시작 후 해결됨.
- 다음 단계(STEP 02) 진행 가능 여부: 가능
- 추가 확인 사항: 새 세션에서 커널이 다시 엉키면, `import sys; print(sys.executable)`로 경로가 `C:\dev\claude-code-agent-course\.venv\Scripts\python.exe`인지 먼저 확인할 것

# STEP 03. 채용공고 페이지 접근 테스트

## 작업 계획

실제 데이터 수집 전에, `requests`로 잡코리아 검색 결과 페이지에 접근했을 때 어떤 응답이 오는지 확인한다.

확인 항목:
- 요청 상태 코드
- 응답 Content-Type
- 응답 길이
- 응답 HTML 안에 실제 채용공고 데이터(회사명, 공고 제목 등)가 들어있는지

아직 여러 페이지를 반복 수집하지 않는다.

In [ ]:
import requests

# 사용자가 브라우저에서 직접 "AX" 검색 후 복사한 실제 URL
url = "https://www.jobkorea.co.kr/Search/?stext=ax&tabType=recruit"
res = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})

print("status code:", res.status_code)
print("content-type:", res.headers.get("content-type"))
print("length:", len(res.text))
print("contains 'jobTitle' text:", "jobTitle" in res.text)
print("contains 'legacyJobNos' text:", "legacyJobNos" in res.text)

## 실행 결과 해석 (1차 - 이후 정정됨)

- 요청 성공 여부: 성공 (상태 코드 200, 응답 길이 348,757자)
- 확인한 데이터: `jobTitle`, `legacyJobNos` 라는 **문자열이 그대로** 있는지만 검색했는데 없었음
- **⚠️ 정정**: 위 결론(SPA라 데이터가 없다)은 **틀렸음**. 실제로는 원본 HTML에 채용공고 데이터가 `<div data-sentry-component="CardJob">` 형태로 그대로 들어있고, 회사명/제목/경력/지역 정보는 `requests + BeautifulSoup`로 정상 추출 가능함을 재확인함. 단순히 "jobTitle"이라는 문자열 자체가 코드에 없었을 뿐, 데이터 자체는 있었음 (검색 방법이 잘못됐던 것)
- 다음 단계 진행 가능 여부: **가능. 실제 크롤러로 STEP 04 진행**
- 추가 확인 사항: 등록일/마감일 정보는 페이지 로딩 후 별도 API(`/Search/api/display/v1/jobs/status`)로 채워지는데, 이 API는 브라우저 세션 헤더가 필요해서 `requests`만으로는 안정적으로 못 가져옴 → 일단 이 두 컬럼은 비워두고 진행, 필요하면 나중에 재도전

# STEP 04. 소량 데이터 수집 (실제 크롤링)

## 작업 계획

STEP 03에서 확인한 실제 HTML 구조(`data-sentry-component="CardJob"`)를 이용해,
"AI" 검색어로 실제 채용공고를 소량(최대 10건) 수집하는 함수를 작성하고 테스트한다.

확인 항목:
- 카드 개수가 0이 아닌지
- company_name, job_title, career, location, job_url이 정상적으로 채워지는지
- posted_date/closing_date는 이번 단계에서는 비워둠 (별도 API 필요, 추후 재도전)

아직 여러 검색어로 확장하지 않는다.

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
from datetime import datetime


def collect_jobs(keyword, max_items=10):
    url = f"https://www.jobkorea.co.kr/Search/?stext={keyword}&tabType=recruit"
    res = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
    res.encoding = "utf-8"
    soup = BeautifulSoup(res.text, "html.parser")
    cards = soup.find_all("div", attrs={"data-sentry-component": "CardJob"})

    rows = []
    for card in cards[:max_items]:
        title_a = card.find("a", attrs={"data-sentry-component": "Title"})
        if not title_a:
            continue
        job_title = title_a.get_text(strip=True)
        job_url = title_a.get("href", "").split("?")[0]

        company_name = None
        title_wrapper = title_a.find_parent("div")
        if title_wrapper:
            company_span_wrapper = title_wrapper.find_next_sibling("span")
            if company_span_wrapper:
                company_a = company_span_wrapper.find("a")
                if company_a:
                    company_span = company_a.find("span", class_="truncate")
                    company_name = company_span.get_text(strip=True) if company_span else None

        chips = card.find_all("div", attrs={"data-sentry-component": "GrayChip"})
        location = None
        if chips:
            loc_span = chips[0].find("span", class_="truncate")
            location = loc_span.get_text(strip=True) if loc_span else None

        full_text = card.get_text(" ", strip=True)
        career_match = re.search(r"(경력\s*\d+년\s*↑|경력무관|신입[·/]?경력?|신입)", full_text)
        career = career_match.group(1) if career_match else None

        rows.append({
            "company_name": company_name,
            "job_title": job_title,
            "career": career,
            "location": location,
            "posted_date": None,
            "closing_date": None,
            "job_url": job_url,
            "search_keyword": keyword,
            "collected_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        })
    return pd.DataFrame(rows)


df = collect_jobs("ax", max_items=10)
print("shape:", df.shape)
df.head(10)

## 실행 결과 해석

- 실행 성공 여부: 성공
- 행/열 개수: (10, 9) — "ax" 검색어로 실제 공고 10건 수집
- company_name / job_title / career / location / job_url이 실제 값으로 채워졌는지: company_name, job_title, location, job_url은 정상 채워짐 (GS리테일, 에스코어, ㈜NAVER 등 실제 회사/공고). career는 일부 행에서 NaN 발생 (정규식이 못 잡은 표현 방식 존재 — STEP 06 전처리에서 결측 처리 예정)
- 예상과 다른 부분: posted_date/closing_date는 계획대로 비어있음(None) — 별도 API 필요, 추후 재도전
- 다음 단계(STEP 05, DataFrame 확정) 진행 가능 여부: 가능

# STEP 04-1. 상세 페이지에서 날짜 정보 보강

## 작업 계획

목록 페이지에는 등록일/마감일이 없었지만, **공고 상세 페이지**의 JSON-LD 구조화 데이터
(`<script type="application/ld+json">`, schema.org `JobPosting` 형식)에
`datePosted`(등록일), `validThrough`(마감일)가 정확하게 들어있음을 확인했다.

이번 단계에서는 `df`의 각 공고 URL에 접속해서 이 두 값을 가져와 채운다.

확인 항목:
- 각 공고마다 추가 요청 1회씩 발생 (10건이면 10회 요청)
- 날짜가 정상적으로 채워지는지
- 요청 실패 시 None으로 두고 넘어가는지 (에러로 중단되지 않도록)

In [ ]:
import json
import time


def get_job_dates(job_url):
    try:
        res = requests.get(job_url, headers={"User-Agent": "Mozilla/5.0"}, timeout=10)
        res.encoding = "utf-8"
        soup = BeautifulSoup(res.text, "html.parser")
        for script in soup.find_all("script", attrs={"type": "application/ld+json"}):
            try:
                data = json.loads(script.string)
            except (TypeError, json.JSONDecodeError):
                continue
            if isinstance(data, dict) and data.get("@type") == "JobPosting":
                posted = data.get("datePosted")
                closing = data.get("validThrough")
                return (posted[:10] if posted else None, closing[:10] if closing else None)
    except requests.RequestException:
        pass
    return None, None


posted_list, closing_list = [], []
for url in df["job_url"]:
    posted, closing = get_job_dates(url)
    posted_list.append(posted)
    closing_list.append(closing)
    time.sleep(0.3)  # 서버 부하를 줄이기 위한 짧은 대기

df["posted_date"] = posted_list
df["closing_date"] = closing_list

print("posted_date 채워진 개수:", df["posted_date"].notna().sum(), "/", len(df))
print("closing_date 채워진 개수:", df["closing_date"].notna().sum(), "/", len(df))
df[["company_name", "job_title", "posted_date", "closing_date"]]

## 실행 결과 해석

- 실행 성공 여부: 성공
- posted_date / closing_date 채워진 개수: 10/10 (전부 성공)
- 실제 날짜 값이 그럴듯한지: 그럴듯함 (예: 2026-09-01 등록 / 2026-10-31 마감처럼 등록일이 마감일보다 앞섬, 최근~가까운 미래 날짜 범위)
- 예상과 다른 부분: 없음. 목록 페이지가 아닌 상세 페이지의 JSON-LD 구조화 데이터에서 가져오는 방식이 안정적으로 동작함
- 다음 단계(STEP 05부터 다시 실행) 진행 가능 여부: 가능

> ⚠️ 이 셀 이후 STEP 05, 06, 07 셀들을 **위에서부터 다시 순서대로 실행**해야
> posted_date/closing_date가 채워진 최신 `df`/`df_clean` 기준으로 갱신됩니다.

# STEP 05. DataFrame 생성/확정

## 작업 계획

STEP 04에서 수집한 DataFrame(`df`)의 컬럼 구성과 데이터 타입을 최종 확인한다.

확인 항목:
- 컬럼 목록이 SPEC.md의 9개 컬럼과 정확히 일치하는지
- 각 컬럼의 데이터 타입 (`df.info()`)
- 결측치 현황 (`df.isna().sum()`)

아직 결측치를 채우거나 제거하지 않는다 (그건 STEP 06).

In [ ]:
expected_columns = [
    "company_name", "job_title", "career", "location",
    "posted_date", "closing_date", "job_url",
    "search_keyword", "collected_at",
]

print("컬럼 일치 여부:", list(df.columns) == expected_columns)
print()
df.info()
print()
print("결측치 현황:")
print(df.isna().sum())

## 실행 결과 해석

- 실행 성공 여부: 성공
- 컬럼 일치 여부: True (SPEC.md 9개 컬럼과 정확히 일치)
- 데이터 타입 이상 여부: 없음. 모든 컬럼이 문자열(str/object)로 되어있음 — posted_date/closing_date는 원래 계획대로 값이 없어 타입 확정은 추후(날짜 데이터 들어올 때) 재검토
- 결측치 현황: career 3/10건 결측, posted_date/closing_date 10/10건 결측(전부 없음), 나머지 컬럼은 결측 없음
- 다음 단계(STEP 06, 전처리/중복 제거) 진행 가능 여부: 가능

# STEP 06. 전처리 / 중복 제거

## 작업 계획

STEP 05에서 확인한 DataFrame(`df`)을 정제한다.

작업 항목:
- `job_url` 기준 중복 공고 확인 및 제거
- `career` 결측치를 "정보없음" 문자열로 채우기
- `posted_date`/`closing_date`는 현재 전부 결측이므로 채우지 않고 그대로 둠 (날짜 없이 진행하기로 결정함)

결과는 `df_clean`이라는 새 변수에 저장한다 (원본 `df`는 그대로 보존).

In [ ]:
dup_count = df.duplicated(subset=["job_url"]).sum()
print("job_url 중복 개수:", dup_count)

df_clean = df.drop_duplicates(subset=["job_url"]).copy()
df_clean["career"] = df_clean["career"].fillna("정보없음")

print("전처리 전:", df.shape)
print("전처리 후:", df_clean.shape)
print()
print("career 결측치 (전처리 후):", df_clean["career"].isna().sum())
df_clean

## 실행 결과 해석

- 실행 성공 여부: 성공
- job_url 중복 개수: 0건
- 전처리 전/후 행 개수 변화: (10, 9) → (10, 9), 중복이 없어 변화 없음
- career 결측치 처리 결과: 3건 → 0건, "정보없음"으로 정상 대체됨
- 다음 단계(STEP 07, 신규 공고 판별) 진행 가능 여부: 가능

# STEP 07. 신규 공고 판별

## 작업 계획

`data/processed/jobs_history.csv`에 지금까지 수집된 공고 URL 이력을 저장하고,
이번에 수집한 `df_clean`과 비교해서 **새로 나타난 공고(job_url 기준)**만 골라낸다.

작업 항목:
- 이력 파일이 없으면 새로 만들고, 있으면 불러온다
- 이번 수집 결과 중 이력에 없는 job_url만 "신규 공고"로 표시
- 이번 실행이 끝나면 전체 이력을 이력 파일에 다시 저장 (다음 주 실행을 위해)

이번이 첫 실행이므로 전부 신규로 처리될 것으로 예상한다.

In [ ]:
import os

HISTORY_PATH = "../data/processed/jobs_history.csv"

if os.path.exists(HISTORY_PATH):
    history_df = pd.read_csv(HISTORY_PATH)
    known_urls = set(history_df["job_url"])
else:
    history_df = pd.DataFrame(columns=df_clean.columns)
    known_urls = set()

df_clean["is_new"] = ~df_clean["job_url"].isin(known_urls)
new_jobs = df_clean[df_clean["is_new"]].copy()

print("이력에 있던 공고 수:", len(known_urls))
print("이번 수집 공고 수:", len(df_clean))
print("신규 공고 수:", len(new_jobs))

# 이력 갱신 (기존 + 이번 수집분 합쳐서 저장, job_url 기준 중복 제거)
updated_history = pd.concat([history_df, df_clean.drop(columns=["is_new"])], ignore_index=True)
updated_history = updated_history.drop_duplicates(subset=["job_url"])
updated_history.to_csv(HISTORY_PATH, index=False)

new_jobs

## 실행 결과 해석

- 실행 성공 여부: 성공
- 이력에 있던 공고 수 / 이번 신규 공고 수: 10 / 1
- 예상(첫 실행이라 전부 신규)과 일치하는지: 불일치하지만 정상. 실제로는 이전 STEP 04 실행 때 이미 이력 파일이 한 번 생성되어 10건이 저장돼 있었고, 이번에 다시 실시간 크롤링하니 실제 사이트 목록이 살짝 바뀌어 9건은 기존과 동일, 1건이 새로 나타난 공고로 정상 감지됨 (실시간 사이트이기 때문에 자연스러운 현상)
- `data/processed/jobs_history.csv` 파일이 정상 생성되었는지: 정상 생성/갱신됨
- 다음 단계(STEP 08, 기본 분석) 진행 가능 여부: 가능

# STEP 08. 기본 분석 / 관련 공고 필터링

## 작업 계획

Gemini API를 호출하기 전에, pandas로 계산 가능한 사실들을 먼저 확인한다
(계산 가능한 사실은 pandas로, Gemini는 요약/해석 전용).

확인 항목:
- 신규 공고 수 (STEP 07 결과 재확인)
- 회사별 공고 수
- 지역별 공고 수
- 경력 조건 분포

In [ ]:
print("신규 공고 수:", len(new_jobs))
print()

print("회사별 공고 수:")
print(df_clean["company_name"].value_counts())
print()

print("지역별 공고 수:")
print(df_clean["location"].value_counts())
print()

print("경력 조건 분포:")
print(df_clean["career"].value_counts())

## 실행 결과 해석

- 실행 성공 여부: 성공
- 신규 공고 수: 0건 (이전 실행에서 이미 이력에 반영된 상태라 이번엔 신규 없음 — 정상)
- 회사별/지역별 분포에서 눈에 띄는 점: ㈜NAVER가 2건으로 가장 많음, 지역은 경기 성남시(㈜NAVER 등)가 3건으로 가장 많음
- 경력 조건 분포에서 눈에 띄는 점: 경력 요건이 다양하게 분포(경력무관~경력8년↑), 특정 연차에 쏠리지 않음
- 다음 단계(STEP 09, Gemini API 연동) 진행 가능 여부: 가능

# STEP 09. Gemini API 연동

## 작업 계획

`.env`에 저장된 `GEMINI_API_KEY`를 읽어와 Gemini API에 연결하고,
`df_clean`의 공고 1건만 선택해서 요약을 요청하는 테스트를 진행한다.

Gemini에게 맡기는 역할은 요약/설명으로 제한한다 (통계 계산은 이미 pandas로 끝냄).

확인 항목:
- API 키가 정상적으로 로드되는지
- Gemini 응답이 정상적으로 오는지
- 응답 내용이 공고 제목과 관련이 있어 보이는지 (본격 검증은 STEP 10에서)

In [ ]:
from dotenv import load_dotenv
from google import genai
import os

load_dotenv("../.env", override=True)
api_key = os.getenv("GEMINI_API_KEY")
print("API 키 로드 여부:", bool(api_key))

client = genai.Client(api_key=api_key)

sample_job = df_clean.iloc[0]
prompt = f"""다음 채용공고를 3줄로 요약해줘.

회사명: {sample_job['company_name']}
공고 제목: {sample_job['job_title']}
경력: {sample_job['career']}
지역: {sample_job['location']}
"""

response = client.models.generate_content(
    model="gemini-flash-lite-latest",
    contents=prompt,
)

print(response.text)

## 실행 결과 해석

- 실행 성공 여부: 성공
- API 키 로드 여부: True
- Gemini 응답 내용이 공고와 관련 있어 보이는지: 그렇다. GS리테일/BIZ CLUB팀/MD AX/물류 AX/경력 3년/서울 강남구 등 실제 입력한 정보가 정확히 반영된 3줄 요약이 나옴
- 예상과 다른 부분: "AFC(자동 함수 호출)" 관련 안내 메시지가 함께 출력되는데, 에러가 아니라 SDK의 권장사항 안내 문구라 무시해도 됨
- 다음 단계(STEP 10, Gemini 결과 검증) 진행 가능 여부: 가능

# STEP 10. Gemini 결과 검증

## 작업 계획

`df_clean`에서 3건을 골라 Gemini로 요약을 생성하고, **원본 데이터(회사명/제목/경력/지역)와
나란히 출력**해서 사람이 직접 비교·검증한다.

확인 항목 (해석 셀에 공고별로 기록):
- 원문에서 확인한 정보
- Gemini가 요약한 내용
- 누락된 부분
- 과도하게 해석/추측한 부분
- 이 요약을 보고서에 그대로 써도 되는지

API 호출 성공 자체가 아니라 **내용의 정확성**을 검증하는 단계임을 유의한다.

In [ ]:
import time
from google.genai.errors import APIError

_summary_cache = {}


def summarize_job(row, retries=3, wait_seconds=25):
    # 같은 공고(job_url)를 이미 요약한 적 있으면 API를 다시 호출하지 않고 캐시를 재사용
    if row["job_url"] in _summary_cache:
        return _summary_cache[row["job_url"]]

    prompt = f"""다음 채용공고를 3줄로 요약해줘.

회사명: {row['company_name']}
공고 제목: {row['job_title']}
경력: {row['career']}
지역: {row['location']}
"""
    for attempt in range(retries):
        try:
            response = client.models.generate_content(
                model="gemini-flash-lite-latest",
                contents=prompt,
            )
            _summary_cache[row["job_url"]] = response.text
            return response.text
        except APIError as e:
            if attempt == retries - 1:
                return f"(Gemini 요약 실패: {e.code} {e.status} - 잠시 후 다시 시도해주세요)"
            time.sleep(wait_seconds)


for i, row in df_clean.head(3).iterrows():
    print("=" * 60)
    print("[원본 데이터]")
    print("회사명:", row["company_name"])
    print("제목:", row["job_title"])
    print("경력:", row["career"])
    print("지역:", row["location"])
    print()
    print("[Gemini 요약]")
    print(summarize_job(row))
    print()
    time.sleep(13)  # 무료 티어 분당 5회 제한을 넘지 않도록 호출 간격 유지

## 실행 결과 해석

검증한 공고 수: 3 (2건 성공, 1건은 서버 과부하로 재시도 후에도 실패 → 안전하게 실패 메시지 처리됨)

### 공고 1 (GS리테일)
- 원문에서 확인한 정보: BIZ CLUB팀/MD AX/물류 AX/보안성 검토 담당, 경력3년↑, 서울 강남구 외 1
- Gemini 요약 내용: 회사/직무/경력/지역이 원문 그대로 정확히 반영됨
- 누락/과잉 해석: 없음
- 사용 가능 여부: 가능

### 공고 2 (에스코어)
- 원문에서 확인한 정보: AX 컨설턴트 채용, 경력 정보없음, 서울 송파구
- Gemini 요약 내용: "경력 요건은 정보 없음 상태"라고 있는 그대로 정확하게 표현함 (있지도 않은 경력 조건을 지어내지 않음)
- 누락/과잉 해석: 없음
- 사용 가능 여부: 가능

### 공고 3
- 원문에서 확인한 정보: (API 서버 과부하로 응답을 받지 못함)
- Gemini 요약 내용: "(Gemini 서버 과부하로 요약 실패 - 잠시 후 다시 시도해주세요)"
- 누락/과잉 해석: 해당 없음 (요약 자체가 생성되지 않음)
- 사용 가능 여부: 이번엔 불가. 실패를 있는 그대로 표시해서 잘못된 정보가 보고서에 들어가는 것은 막았음(의도한 동작)

- 다음 단계(STEP 11, Markdown 보고서 생성) 진행 가능 여부: 가능. 다만 실제 운영에서는 재시도 횟수/대기시간을 늘리거나 실패 건을 재수집하는 로직이 추가로 필요할 수 있음

# STEP 11. Markdown 보고서 생성

## 작업 계획

SPEC.md의 보고서 템플릿(요약 / 주요 동향 / 추천 공고 / 데이터 기준 / 주의사항)에 맞춰
`reports/weekly_report_YYYY-MM-DD.md` 파일을 생성한다.

- 사실(통계)은 pandas로 계산한 값을 사용
- "추천 공고" 항목은 상위 3건에 대해 Gemini 요약을 붙임 (STEP 10에서 검증한 방식 재사용)
- 신규 공고가 없을 경우, 이번 수집 결과 상위 3건으로 대체 표시

In [ ]:
from datetime import date

report_date = date.today().isoformat()
highlight_jobs = new_jobs if len(new_jobs) > 0 else df_clean.head(3)

top_companies = df_clean["company_name"].value_counts().head(5)
top_locations = df_clean["location"].value_counts().head(5)

lines = []
lines.append("# 주간 AX 채용 동향")
lines.append(f"실행일: {report_date}")
lines.append("")
lines.append("## 1. 이번 주 요약")
lines.append(f"- 이번 수집 공고 수: {len(df_clean)}건")
lines.append(f"- 신규 공고: {len(new_jobs)}건")
lines.append(f"- 검색 키워드: {df_clean['search_keyword'].unique().tolist()}")
lines.append("")
lines.append("## 2. 주요 동향")
lines.append("- 회사별 공고 수 Top 5:")
for company, count in top_companies.items():
    lines.append(f"  - {company}: {count}건")
lines.append("- 지역별 공고 수 Top 5:")
for loc, count in top_locations.items():
    lines.append(f"  - {loc}: {count}건")
lines.append("")
lines.append("## 3. 추천 공고")
for idx, (_, row) in enumerate(highlight_jobs.head(3).iterrows(), start=1):
    lines.append(f"### {idx}) {row['company_name']} / {row['job_title']}")
    lines.append(f"- 경력: {row['career']}")
    lines.append(f"- 지역: {row['location']}")
    lines.append(f"- 등록일/마감일: {row['posted_date']} ~ {row['closing_date']}")
    lines.append(f"- Gemini 요약: {summarize_job(row)}")
    lines.append(f"- 공고 링크: {row['job_url']}")
    lines.append("")
    time.sleep(13)  # 무료 티어 분당 5회 제한을 넘지 않도록 호출 간격 유지
lines.append("## 4. 데이터 기준")
lines.append(f"- 검색어: {df_clean['search_keyword'].unique().tolist()}")
lines.append(f"- 수집 시각: {df_clean['collected_at'].iloc[0]}")
lines.append(f"- 분석 대상 건수: {len(df_clean)}건")
lines.append("")
lines.append("## 5. 주의사항")
lines.append("- 실제 지원 전 원문 공고를 다시 확인할 것")
lines.append("- 이 보고서는 학습용 파이프라인 실습 결과이며, Gemini 요약은 참고용임")

report_text = "\n".join(lines)

report_path = f"../reports/weekly_report_{report_date}.md"
with open(report_path, "w", encoding="utf-8") as f:
    f.write(report_text)

print("저장 위치:", report_path)
print()
print(report_text)

## 실행 결과 해석

- 실행 성공 여부: 성공
- 파일이 정상 생성되었는지: `reports/weekly_report_2026-09-23.md` 정상 생성 확인
- 보고서 내용이 SPEC.md 템플릿과 맞는지: 일치 (요약/주요동향/추천공고 3건/데이터기준/주의사항 모두 포함)
- 예상과 다른 부분: 모델 일일 할당량(`gemini-3.6-flash` 하루 20회) 소진 이슈가 있었으나 `gemini-flash-lite-latest`로 교체 + 캐싱 적용 후 3건 모두 정상 요약됨
- 다음 단계(STEP 12, Slack 발송) 진행 가능 여부: 가능

# STEP 12. Slack 발송

## 작업 계획

STEP 11에서 생성한 보고서(`report_text`)를 `.env`의 `SLACK_WEBHOOK_URL`로 전송한다.

확인 항목:
- HTTP 상태 코드 (200/ok)
- 실제 Slack 채널에 메시지가 도착하는지
- 한글이 깨지지 않는지
- 메시지 길이가 너무 길어서 잘리지 않는지

In [ ]:
from dotenv import load_dotenv

load_dotenv("../.env", override=True)


def send_slack(text):
    webhook_url = os.getenv("SLACK_WEBHOOK_URL")
    if not webhook_url:
        print("SLACK_WEBHOOK_URL이 설정되지 않았습니다.")
        return False
    res = requests.post(webhook_url, json={"text": text})
    print("status:", res.status_code, "response:", res.text)
    return res.status_code == 200 and res.text == "ok"


success = send_slack(report_text)
print("발송 성공 여부:", success)

## 실행 결과 해석

- HTTP 상태 / 응답: 200 / ok
- Slack 채널에 실제 도착 여부: 도착 확인됨
- 한글 깨짐 여부: 없음
- 메시지 길이/가독성 문제: 없음. 보고서 전체(요약/동향/추천공고 3건/데이터기준/주의사항)가 잘리지 않고 그대로 도착함
- 다음 단계(STEP 13, Gmail 발송) 진행 가능 여부: 가능

> 참고: 이전에 `SLACK_WEBHOOK_URL이 설정되지 않았습니다` 에러가 있었는데,
> `load_dotenv()`가 이미 로드된 빈 값을 덮어쓰지 않아서 발생한 문제였음.
> `load_dotenv("../.env", override=True)`로 해결함 — `.env` 값을 실행 중간에
> 새로 채워 넣었을 때는 항상 `override=True`로 다시 불러와야 함.

# STEP 13. Gmail 발송

## 작업 계획

STEP 11에서 생성한 보고서(`report_text`)를 `.env`의 `GMAIL_USER`/`GMAIL_APP_PASSWORD`로
본인 Gmail 앞으로 이메일 발송한다 (`smtplib` 사용, 앱 비밀번호 인증).

확인 항목:
- 발송 성공 여부 (에러 없이 완료되는지)
- 실제 받은편지함에 도착하는지
- 제목/본문이 정상적으로 보이는지

In [ ]:
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

load_dotenv("../.env", override=True)
gmail_user = os.getenv("GMAIL_USER")
gmail_app_password = os.getenv("GMAIL_APP_PASSWORD")


def send_email(subject, body, to_addr=None):
    to_addr = to_addr or gmail_user
    if not gmail_user or not gmail_app_password:
        print("GMAIL_USER 또는 GMAIL_APP_PASSWORD가 설정되지 않았습니다.")
        return False

    msg = MIMEMultipart()
    msg["From"] = gmail_user
    msg["To"] = to_addr
    msg["Subject"] = subject
    msg.attach(MIMEText(body, "plain"))

    try:
        with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
            server.login(gmail_user, gmail_app_password)
            server.sendmail(gmail_user, to_addr, msg.as_string())
        return True
    except smtplib.SMTPException as e:
        print("Gmail 발송 실패:", e)
        return False


success = send_email(f"주간 AX 채용 동향 ({report_date})", report_text)
print("발송 성공 여부:", success)

## 실행 결과 해석

- 발송 성공 여부: 성공
- 받은편지함 도착 여부: 도착 확인됨
- 제목/본문 표시 문제: 없음. 제목("주간 AX 채용 동향 (2026-09-23)")과 본문(보고서 전체) 모두 정상 표시됨
- 다음 단계(STEP 14, 함수화) 진행 가능 여부: 가능

# STEP 14. 함수화

## 작업 계획

지금까지 이 노트북의 셀에 흩어져 있던 코드를 `src/` 폴더의 재사용 가능한 함수로 옮긴다.

- `src/crawler.py` — `collect_jobs()`, `get_job_dates()`, `enrich_with_dates()`
- `src/preprocess.py` — `clean_jobs()`, `find_new_jobs()`
- `src/analyzer.py` — `analyze_jobs()`
- `src/gemini_client.py` — `get_gemini_client()`, `summarize_job()`
- `src/reporter.py` — `create_report()`
- `src/notifier.py` — `send_slack()`, `send_email()`

Notebook은 버리지 않는다 — 앞으로도 셀 단위 검증 기록으로 계속 사용하고,
`src/`는 `main.py`(STEP 15)에서 재사용할 운영 코드가 된다.

In [ ]:
import sys

sys.path.insert(0, "..")

from src.crawler import collect_jobs, enrich_with_dates
from src.preprocess import clean_jobs, find_new_jobs
from src.analyzer import analyze_jobs
from src.gemini_client import get_gemini_client, summarize_job
from src.reporter import create_report
from src.notifier import send_slack, send_email

print("src 모듈 import 성공")

# 기존 이력 데이터로 preprocess/analyzer 함수만 빠르게 재검증 (API 재호출 없음)
history_df = pd.read_csv("../data/processed/jobs_history.csv")
df_clean_v2 = clean_jobs(history_df)
sample_new_jobs = df_clean_v2.head(2)
analysis_v2 = analyze_jobs(df_clean_v2, sample_new_jobs)

print("total_count:", analysis_v2["total_count"])
print("new_count:", analysis_v2["new_count"])
print(analysis_v2["top_companies"])

## 실행 결과 해석

- 실행 성공 여부: 성공 (Claude Code가 Bash에서도 동일하게 사전 검증함)
- src 모듈 import 및 함수 동작 확인: `clean_jobs()`, `analyze_jobs()`가 저장된 이력 데이터 기준으로 정상 동작함
- 다음 단계(STEP 15, main.py 통합) 진행 가능 여부: 가능

# STEP 15. main.py 통합

## 작업 계획

프로젝트 루트에 `main.py`를 작성해서, `src/`의 함수들을 아래 순서로 호출하는
전체 흐름만 담당하게 한다 (세부 로직은 `main.py`에 직접 두지 않음).

```
collect_jobs → enrich_with_dates → clean_jobs → find_new_jobs
→ analyze_jobs → create_report → send_slack → send_email
```

이 셀에서는 아직 `main.py`를 실행하지 않는다 (실행하면 실제 크롤링/Gemini 호출/
Slack·Gmail 발송이 일어남) — 실제 실행은 STEP 16에서 진행한다.

In [164]:
with open("../main.py", encoding="utf-8") as f:
    print(f.read())

from datetime import datetime

from src.crawler import collect_jobs, enrich_with_dates
from src.preprocess import clean_jobs, find_new_jobs
from src.analyzer import analyze_jobs
from src.gemini_client import get_gemini_client
from src.reporter import create_report
from src.notifier import send_slack, send_email

SEARCH_KEYWORD = "ax"
MAX_ITEMS = 10


def main():
    start_time = datetime.now()
    print(f"[{start_time}] AX Job Agent 실행 시작 (키워드: {SEARCH_KEYWORD})")

    jobs = collect_jobs(SEARCH_KEYWORD, max_items=MAX_ITEMS)
    print(f"수집 건수: {len(jobs)}")

    jobs = enrich_with_dates(jobs)
    clean = clean_jobs(jobs)
    new_jobs = find_new_jobs(clean)
    print(f"신규 공고 수: {len(new_jobs)}")

    analysis = analyze_jobs(clean, new_jobs)

    client = get_gemini_client()
    report_path, report_text = create_report(clean, new_jobs, analysis, client)
    print(f"보고서 생성: {report_path}")

    slack_ok = send_slack(report_text)
    print(f"Slack 발송 성공 여부: {slack_ok}")

    email_ok = sen

## 실행 결과 해석

- main.py 내용이 계획한 흐름과 일치하는지: 일치 (collect → enrich → clean → find_new → analyze → report → slack → email)
- 문법 검사(`py_compile`) 결과: 통과 (Claude Code가 Bash로 사전 확인함)
- 다음 단계(STEP 16, 로컬 전체 실행 검증) 진행 가능 여부: 가능 (실제 실행 시 Slack/Gmail 발송 및 Gemini 호출이 발생하므로 준비된 상태에서 진행)

# STEP 16. 로컬 전체 실행 검증

## 작업 계획

터미널에서 `python main.py`를 실제로 실행해서, 로컬 파이프라인 전체가
처음부터 끝까지 에러 없이 성공하는지 확인한다 (Claude Code가 Bash에서 실행).

## 실행 결과 (Claude Code가 터미널에서 실행)

```
수집 건수: 10
신규 공고 수: 0
보고서 생성: reports\weekly_report_2026-09-23.md
Slack 발송 성공 여부: True
Gmail 발송 성공 여부: True
(소요 시간: 약 20초)
```

## 실행 결과 해석

- 실행 성공 여부: 성공, 에러 없이 끝까지 완료
- Slack/Gmail 실제 도착 여부: 사용자가 직접 확인 — 둘 다 정상 도착
- SPEC.md "완료 정의" 체크리스트: 대부분 충족 (수집 0건 아님, 필수 컬럼 존재, 신규판별 정상, 보고서 생성, Slack/Gmail 도착, 에러 시 성공으로 보이지 않음)
- 다음 단계(STEP 17, GitHub Actions 수동 실행) 진행 가능 여부: 가능

# STEP 17. GitHub Actions 수동 실행

## 작업 계획

`.github/workflows/ax-job-agent.yml`을 작성해서 `workflow_dispatch`(수동 실행)만 우선 추가하고,
GitHub Secrets에 `GEMINI_API_KEY`/`SLACK_WEBHOOK_URL`/`GMAIL_USER`/`GMAIL_APP_PASSWORD`를 등록한 뒤,
main 브랜치로 PR 병합 후 실제로 수동 실행해본다.

## 실행 결과 (Claude Code가 `gh` CLI로 직접 실행/확인)

- 1차 실행(main, 최초 push 직후): **성공** — 수집 10건, Slack/Gmail 발송 True
- 2차 실행(사용자가 웹 화면에서 직접 트리거): **실패** — `requests.exceptions.ConnectTimeout`
  (GitHub Actions 러너에서 jobkorea.co.kr 접속이 간헐적으로 타임아웃됨)
- 조치: `src/crawler.py`에 재시도(retry + backoff) 로직 추가 (PR #3)
- 3차 실행(수정 후): **성공** — 수집 10건, Slack/Gmail 발송 True

## 실행 결과 해석

- 실행 성공 여부: 최종적으로 성공 (중간에 1회 실패, 원인 파악 후 재시도 로직으로 해결)
- 원인: 클라우드 CI 환경(GitHub Actions)에서 외부 사이트 접속이 간헐적으로 실패할 수 있음 —
  로컬 실행보다 CI 환경이 더 불안정할 수 있다는 걸 실제로 경험함
- `workflow_dispatch`가 GitHub Actions 화면 목록에 안 보였던 이유: default 브랜치(main)에
  워크플로 파일이 없었기 때문 (feature 브랜치에만 있으면 UI 목록에 안 뜸) → PR 병합으로 해결
- 다음 단계(STEP 18, 주 1회 자동 실행) 진행 가능 여부: 가능

# STEP 18. GitHub Actions 주간 실행 (마지막 단계)

## 작업 계획

`.github/workflows/ax-job-agent.yml`에 `schedule: cron: "0 0 * * 1"`
(UTC 매주 월요일 00:00 = 한국시간 09:00)을 추가해서, 매주 자동으로
`main.py`가 실행되도록 한다. `schedule` 트리거는 반드시 default 브랜치(main)에
있어야 GitHub이 인식하므로, 추가 후 바로 main으로 PR 병합한다.

## 실행 결과 (Claude Code가 GitHub API로 확인)

main 브랜치의 워크플로 파일에 `schedule` 항목이 정상적으로 반영됨을 확인함
(`gh api`로 파일 내용 직접 조회).

## 실행 결과 해석

- 설정 성공 여부: 성공
- 실제 스케줄 발동 확인: cron은 미래 시점에만 실행되므로, 다음 월요일 09:00(KST) 이후
  GitHub Actions 실행 기록(Actions 탭)에서 자동 실행 여부를 확인하면 됨
- 완료 여부: **강의안 STEP 00~18 전체 완주**. 수집(실제 크롤링) → 정제 →
  분석 → Gemini 요약 → Markdown 보고서 → Slack/Gmail 발송 → GitHub Actions
  자동화까지 전체 파이프라인이 로컬과 클라우드(GitHub Actions) 양쪽에서 모두 동작 확인됨
- 향후 개선 아이디어 (선택): 여러 검색어로 확장, posted_date/closing_date 실패 시
  방어 코드 보강, 신규 공고만 필터링해 Gemini 요약 대상 축소(비용 절감)